#Xp ninja cnn

In [ ]:
# Exercises XP Ninja : Cinq projets CNN avancés
# Techniques avancées : multi-label, imbalance, transfer learning, Grad-CAM, pipeline

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10, mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

EXERCICE 1 : Classification multi-label

In [ ]:
# EXERCICE 1 : Classification multi-label
# Objectif : une image peut appartenir à plusieurs classes en même temps

# Simulation d'un dataset multi-label à partir de CIFAR-10
# On crée des labels supplémentaires basés sur la couleur dominante des images
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# Créer un deuxième label : image claire (1) ou sombre (0)
# basé sur la luminosité moyenne de l'image
brightness_train = x_train.mean(axis=(1, 2, 3))
brightness_test  = x_test.mean(axis=(1, 2, 3))

label_bright_train = (brightness_train > 0.5).astype(float)
label_bright_test  = (brightness_test  > 0.5).astype(float)

# Labels multi-label : classe originale + label de luminosité
y_train_class = to_categorical(y_train, 10)
y_test_class  = to_categorical(y_test,  10)

y_train_multilabel = np.hstack([
    y_train_class,
    label_bright_train.reshape(-1, 1)
])
y_test_multilabel = np.hstack([
    y_test_class,
    label_bright_test.reshape(-1, 1)
])

# Modèle multi-label
# La sortie utilise sigmoid car chaque label est indépendant
model_ex1 = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  input_shape=(32, 32, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    # Sigmoid pour multi-label : chaque neurone prédit indépendamment
    layers.Dense(11, activation='sigmoid')
])

# binary_crossentropy car chaque label est un problème binaire indépendant
model_ex1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_ex1 = model_ex1.fit(
    x_train, y_train_multilabel,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

_, acc_ex1 = model_ex1.evaluate(x_test, y_test_multilabel, verbose=0)
print(f"Exercice 1 - Accuracy multi-label : {acc_ex1*100:.2f}%")

EXERCICE 2 : Gestion du déséquilibre de classes

In [ ]:
# EXERCICE 2 : Gestion du déséquilibre de classes
# Objectif : traiter un dataset où certaines classes sont rares

print("\nExercice 2 : Gestion du déséquilibre de classes")

# Simuler un déséquilibre : garder 100% des chats (classe 3)
# mais seulement 10% des autres classes
cat_class = 3
mask_cat   = (y_train.ravel() == cat_class)
mask_other = (y_train.ravel() != cat_class)

# Sous-échantillonner les classes autres que chat
other_indices = np.where(mask_other)[0]
cat_indices   = np.where(mask_cat)[0]
selected_other = np.random.choice(
    other_indices,
    size=int(len(other_indices) * 0.1),
    replace=False
)

imbalanced_idx = np.concatenate([cat_indices, selected_other])
np.random.shuffle(imbalanced_idx)

x_imb = x_train[imbalanced_idx]
y_imb = y_train[imbalanced_idx].ravel()

print(f"Total images : {len(y_imb)}")
print(f"Chats (classe 3) : {(y_imb == cat_class).sum()}")
print(f"Autres classes   : {(y_imb != cat_class).sum()}")

# Calculer les poids de classe pour compenser le déséquilibre
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_imb),
    y=y_imb
)
class_weight_dict = dict(enumerate(class_weights_array))
print("Poids des classes :", class_weight_dict)

y_imb_enc = to_categorical(y_imb, 10)

def build_small_cnn(input_shape=(32, 32, 3), num_classes=10):
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu',
                      padding='same', input_shape=input_shape),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Modèle sans gestion du déséquilibre
model_no_balance = build_small_cnn()
model_no_balance.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_no_balance.fit(
    x_imb, y_imb_enc,
    epochs=10, batch_size=32,
    validation_split=0.1, verbose=0
)

# Modèle avec poids de classe
model_balanced = build_small_cnn()
model_balanced.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_balanced.fit(
    x_imb, y_imb_enc,
    epochs=10, batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    verbose=0
)

y_test_enc = to_categorical(y_test, 10)
_, acc_no_balance  = model_no_balance.evaluate(x_test, y_test_enc, verbose=0)
_, acc_balanced    = model_balanced.evaluate(x_test, y_test_enc, verbose=0)

print(f"Sans gestion déséquilibre : {acc_no_balance*100:.2f}%")
print(f"Avec poids de classe      : {acc_balanced*100:.2f}%")

# Rapport de classification détaillé pour voir les performances par classe
y_pred_balanced = np.argmax(
    model_balanced.predict(x_test, verbose=0), axis=1
)
print("\nRapport détaillé (modèle équilibré) :")
CLASS_NAMES = [
    'avion', 'voiture', 'oiseau', 'chat', 'cerf',
    'chien', 'grenouille', 'cheval', 'bateau', 'camion'
]
print(classification_report(y_test.ravel(), y_pred_balanced,
                             target_names=CLASS_NAMES))

EXERCICE 3 : Transfer Learning et fine-tuning avancé

In [ ]:
# EXERCICE 3 : Transfer Learning et fine-tuning avancé
# Objectif : comparer différentes stratégies de fine-tuning

print("\nExercice 3 : Transfer Learning et fine-tuning")

def build_transfer_model(base_model, num_classes=10, trainable_layers=0):
    # Geler toutes les couches d'abord
    base_model.trainable = True
    for layer in base_model.layers[:-trainable_layers]:
        layer.trainable = False

    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

In [ ]:
# Stratégie 1 : geler tout le modèle de base
base1 = keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False, weights='imagenet'
)
model_frozen = build_transfer_model(base1, trainable_layers=0)
model_frozen.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_frozen.fit(
    x_train, y_train_enc,
    epochs=5, batch_size=64,
    validation_split=0.1, verbose=0
)
_, acc_frozen = model_frozen.evaluate(x_test, y_test_enc, verbose=0)

In [ ]:
# Stratégie 2 : dégeler les 20 dernières couches
base2 = keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False, weights='imagenet'
)
model_partial = build_transfer_model(base2, trainable_layers=20)
model_partial.compile(
    optimizer=keras.optimizers.Adam(0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_partial.fit(
    x_train, y_train_enc,
    epochs=5, batch_size=64,
    validation_split=0.1, verbose=0
)
_, acc_partial = model_partial.evaluate(x_test, y_test_enc, verbose=0)

print(f"Modèle gelé (frozen)           : {acc_frozen*100:.2f}%")
print(f"Fine-tuning 20 dernières couches : {acc_partial*100:.2f}%")

EXERCICE 4 : Grad-CAM pour l'interprétabilité

In [ ]:
# EXERCICE 4 : Grad-CAM pour l'interprétabilité
# Objectif : visualiser les zones de l'image qui influencent la décision

print("\nExercice 4 : Grad-CAM - Visualisation de l'attention du modèle")

# Entraîner un modèle simple sur MNIST pour une démo claire
(x_mnist, y_mnist), (x_mnist_test, y_mnist_test) = mnist.load_data()
x_mnist      = x_mnist.reshape(-1, 28, 28, 1) / 255.0
x_mnist_test = x_mnist_test.reshape(-1, 28, 28, 1) / 255.0
y_mnist_enc  = to_categorical(y_mnist, 10)

# Modèle avec une couche nommée pour Grad-CAM
inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D(2, 2)(x)
x = layers.Conv2D(64, (3, 3), activation='relu',
                  padding='same', name='last_conv')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

model_gradcam = keras.Model(inputs, outputs)
model_gradcam.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_gradcam.fit(
    x_mnist, y_mnist_enc,
    epochs=5, batch_size=64, verbose=0
)


In [ ]:
def compute_gradcam(model, img, layer_name, class_idx):
    # Grad-CAM calcule le gradient de la classe cible
    # par rapport aux activations de la dernière couche de convolution
    grad_model = keras.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        img_tensor = tf.cast(img[np.newaxis, ...], tf.float32)
        conv_outputs, predictions = grad_model(img_tensor)
        # Score de la classe cible
        loss = predictions[:, class_idx]

    # Gradients par rapport aux activations convolutionnelles
    grads = tape.gradient(loss, conv_outputs)

    # Importance de chaque canal : moyenne des gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Pondérer les activations par leur importance
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normaliser entre 0 et 1
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

In [ ]:
 #Visualiser Grad-CAM sur quelques images de test
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
sample_imgs  = x_mnist_test[:3]
sample_labels = y_mnist_test[:3]

for i, (img, true_label) in enumerate(zip(sample_imgs, sample_labels)):
    pred = model_gradcam.predict(img[np.newaxis, ...], verbose=0)
    pred_class = np.argmax(pred)
    heatmap = compute_gradcam(model_gradcam, img, 'last_conv', pred_class)

    # Redimensionner la heatmap à la taille de l'image
    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis], (28, 28)
    ).numpy().squeeze()

    # Image originale
    axes[i, 0].imshow(img.squeeze(), cmap='gray')
    axes[i, 0].set_title(f'Original\nLabel: {true_label}')
    axes[i, 0].axis('off')

    # Heatmap Grad-CAM
    axes[i, 1].imshow(heatmap_resized, cmap='hot')
    axes[i, 1].set_title(f'Grad-CAM\nPrédit: {pred_class}')
    axes[i, 1].axis('off')

    # Superposition image + heatmap
    axes[i, 2].imshow(img.squeeze(), cmap='gray', alpha=0.7)
    axes[i, 2].imshow(heatmap_resized, cmap='hot', alpha=0.5)
    axes[i, 2].set_title('Superposition')
    axes[i, 2].axis('off')

    # Distribution des probabilités
    axes[i, 3].bar(range(10), pred[0])
    axes[i, 3].set_title('Probabilités')
    axes[i, 3].set_xlabel('Classe')

plt.suptitle('Grad-CAM : zones d\'attention du modèle', fontsize=12)
plt.tight_layout()
plt.show()


EXERCICE 5 : Pipeline de classification complet

In [ ]:
# EXERCICE 5 : Pipeline de classification complet
# Objectif : créer un pipeline modulaire, réutilisable et robuste

print("\nExercice 5 : Pipeline de classification complet")

class ClassificationPipeline:
    # Pipeline modulaire pour la classification d'images
    # Peut être réutilisé et étendu facilement

    def __init__(self, input_shape, num_classes, model_path='model.h5'):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model_path  = model_path
        self.model       = None
        self.history     = None

    def load_and_preprocess(self, x, y, normalize=True, one_hot=True):
        # Normaliser les pixels si demandé
        if normalize:
            x = x / 255.0

        # Encoder les labels si demandé
        if one_hot:
            y = to_categorical(y, self.num_classes)

        return x, y

    def augment_data(self):
        # Retourne une couche d'augmentation à intégrer au modèle
        return keras.Sequential([
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
        ])

    def build(self, use_augmentation=True):
        # Construire le modèle CNN
        model_layers = []

        if use_augmentation:
            model_layers.append(self.augment_data())

        model_layers.extend([
            layers.Conv2D(32, (3, 3), activation='relu',
                          padding='same', input_shape=self.input_shape),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2, 2),
            layers.Dropout(0.25),

            layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2, 2),
            layers.Dropout(0.25),

            layers.Flatten(),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.4),
            layers.Dense(self.num_classes, activation='softmax')
        ])

        self.model = keras.Sequential(model_layers)
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        return self
    def train(self, x_train, y_train, epochs=20, batch_size=64,
              val_split=0.1, use_callbacks=True):
        # Entraîner le modèle avec callbacks optionnels
        callbacks = []
        if use_callbacks:
            callbacks = [
                keras.callbacks.EarlyStopping(
                    monitor='val_accuracy',
                    patience=5,
                    restore_best_weights=True
                ),
                keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.5,
                    patience=3
                )
            ]

        self.history = self.model.fit(
            x_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=val_split,
            callbacks=callbacks,
            verbose=1
        )
        return self

    def evaluate(self, x_test, y_test):
        # Évaluer et afficher les métriques
        loss, acc = self.model.evaluate(x_test, y_test, verbose=0)
        print(f"Loss     : {loss:.4f}")
        print(f"Accuracy : {acc*100:.2f}%")

        y_pred = np.argmax(self.model.predict(x_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)

        print("\nRapport de classification :")
        print(classification_report(y_true, y_pred))
        return loss, acc

    def plot_history(self):
        # Afficher les courbes d'entraînement
        if self.history is None:
            print("Le modèle n'a pas encore été entraîné.")
            return

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.plot(self.history.history['loss'],     label='Train')
        ax1.plot(self.history.history['val_loss'], label='Validation')
        ax1.set_title('Loss')
        ax1.set_xlabel('Epoch')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(self.history.history['accuracy'],     label='Train')
        ax2.plot(self.history.history['val_accuracy'], label='Validation')
        ax2.set_title('Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.suptitle('Courbes d\'entraînement')
        plt.tight_layout()
        plt.show()

    def save(self):
        # Sauvegarder le modèle
        self.model.save(self.model_path)
        print(f"Modèle sauvegardé : {self.model_path}")

    def load(self):
        # Charger un modèle sauvegardé
        self.model = keras.models.load_model(self.model_path)
        print(f"Modèle chargé : {self.model_path}")
        return self

    def predict(self, image):
        # Prédire la classe d'une image (avec vérification)
        if image is None:
            raise ValueError("L'image ne peut pas être None.")
        if len(image.shape) == 3:
            image = image[np.newaxis, ...]  # Ajouter la dimension batch
        if image.max() > 1.0:
            image = image / 255.0           # Normaliser si nécessaire
        probs = self.model.predict(image, verbose=0)
        return np.argmax(probs, axis=1), probs



In [ ]:
# Utilisation du pipeline
pipeline = ClassificationPipeline(
    input_shape=(32, 32, 3),
    num_classes=10,
    model_path='pipeline_model.h5'
)

In [ ]:
# Charger les données
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = cifar10.load_data()


In [ ]:
# Prétraitement
x_tr, y_tr = pipeline.load_and_preprocess(x_train_raw, y_train_raw)
x_te, y_te = pipeline.load_and_preprocess(x_test_raw,  y_test_raw)


In [ ]:
# Construction et entraînement
pipeline.build(use_augmentation=True)
pipeline.train(x_tr, y_tr, epochs=15, batch_size=64)


In [ ]:
# Évaluation
print("\nÉvaluation finale du pipeline :")
pipeline.evaluate(x_te, y_te)
pipeline.plot_history()


In [ ]:
# Sauvegarde
pipeline.save()

In [ ]:
# Test de prédiction sur une image
pred_class, probs = pipeline.predict(x_te[0])
print(f"\nPrédiction sur l'image 0 : {CLASS_NAMES[pred_class[0]]}")
print(f"Probabilité              : {probs[0][pred_class[0]]*100:.1f}%")